# Sandbox
# 0. 介绍

**研究背景**：Agent 不只会生成文字，还会通过工具执行代码和命令，从而读取文件、修改工作区或启动进程。为了让这些操作真正发生，外层程序必须为 Agent 提供执行环境，并明确操作可以影响的范围。

**现存问题**：如果代码和命令直接在宿主工作区运行，它们就会与真实文件和进程共享同一个环境。这样，即使大模型给出的操作能够完成任务，也可能同时覆盖原文件、留下临时文件或长时间占用资源；一旦操作出错，影响还可能扩散到任务范围之外。

**解决方案**：本 Notebook 将实现一个极简的 Sandbox，把每次任务放进独立的`一次性工作区`，限制执行时间，记录执行前后的文件变化，并且只导出允许保留的产物。然后用同一份真实 API 操作进行对比：基线版本直接修改宿主工作区并造成污染，改进版本在隔离工作区完成相同任务后销毁环境，从而直观看到执行边界如何限制副作用并保持宿主工作区不变。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 说明可用工具
大模型需要先知道自己可以执行什么操作。本节提供一个 `run_python` 工具，告诉大模型可以提交一段 Python 代码；代码将在后面的执行环境中运行。

In [2]:
tools = [{
    "type": "function",
    "function": {
        "name": "run_python",
        "description": "在当前工作区运行一段 Python 代码",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string"}
            },
            "required": ["code"],
        },
    },
}]
print(f"可用工具：{tools[0]['function']['name']}")

可用工具：run_python


输出显示 `run_python` 已经准备好，说明大模型知道了工具名称和需要填写的代码。下一步会给大模型一项具体的文件处理任务。

## 2.2 写出具体任务
为了同时观察任务结果和执行副作用，本节要求大模型读取两个整数，先生成一个计算过程文件，再写出最终结果。真正需要交付的只有 `result.txt`，`scratch.txt` 只是执行过程中的临时文件。

In [3]:
messages = [
    {
        "role": "system",
        "content": "你是文件处理助手，只能调用 run_python。",
    },
    {
        "role": "user",
        "content": (
            "读取 numbers.txt 中的两个整数，先把计算过程写入 scratch.txt，"
            "再把两数之和写入 result.txt。只交付 result.txt。"
        ),
    },
]
print(f"任务：{messages[-1]['content']}")

任务：读取 numbers.txt 中的两个整数，先把计算过程写入 scratch.txt，再把两数之和写入 result.txt。只交付 result.txt。


输出显示了大模型将要完成的任务。这个任务会产生一个最终文件和一个临时文件，因此可以直接观察执行范围是否受到 Sandbox 限制。下一步会准备两份初始状态完全相同的工作区。

## 2.3 准备相同的工作区
### 2.3.1 创建所需目录
基线版本和改进版本需要各自运行，不能互相影响。本节先创建两份独立工作区，再创建一个单独目录，用来接收 Sandbox 最终允许保留的产物。

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

# 所有演示文件都放在同一个临时根目录中
demo_directory = TemporaryDirectory(prefix="nano_sandbox_")
demo_root = Path(demo_directory.name)

# 两条执行路径使用不同工作区，改进版本另有产物目录
baseline_workspace = demo_root / "baseline_workspace"
fixed_workspace = demo_root / "fixed_workspace"
artifact_directory = demo_root / "artifacts"

baseline_workspace.mkdir()
fixed_workspace.mkdir()
artifact_directory.mkdir()
print("已创建：基线工作区、改进工作区、产物目录")

已创建：基线工作区、改进工作区、产物目录


输出说明三个目录已经分开。此时它们都是空的，还没有任务输入；下一步会向两份工作区写入完全相同的数据。

### 2.3.2 写入相同输入
为了确保后面的差异只来自执行方式，本节把同一份 `numbers.txt` 分别写入两份工作区。文件中的两个整数是 `20` 和 `22`。

In [5]:
input_text = "20\n22\n"
workspaces = [baseline_workspace, fixed_workspace]

# 两份工作区写入完全相同的输入文件
for workspace in workspaces:
    input_path = workspace / "numbers.txt"
    input_path.write_text(input_text, encoding="utf-8")

print(f"写入的两个整数：{input_text.split()}")

写入的两个整数：['20', '22']


输出显示两个整数已经写入。基线版本和改进版本现在拥有相同输入；下一步会查看两份工作区的文件列表。

### 2.3.3 查看初始状态
准备完成后，需要把共同起点直接展示出来。本节分别列出两份工作区中的文件，确认它们此时都只有 `numbers.txt`。

In [6]:
# 逐个收集基线工作区中的文件名
baseline_files = []
for path in baseline_workspace.iterdir():
    baseline_files.append(path.name)

# 逐个收集改进工作区中的文件名
fixed_files = []
for path in fixed_workspace.iterdir():
    fixed_files.append(path.name)

print(f"基线工作区：{sorted(baseline_files)}")
print(f"改进工作区：{sorted(fixed_files)}")

基线工作区：['numbers.txt']
改进工作区：['numbers.txt']


输出显示两份工作区都只有同一个输入文件，说明它们的初始状态相同。此时还没有运行大模型生成的代码；下一步会规定两条路径共同使用的成功标准。

## 2.4 定义成功标准
任务成功不能只看答案是否正确，还要看临时文件是否留在原工作区。本节规定：交付文件内容必须是 `42`，原工作区还必须只包含最初的 `numbers.txt`。

In [7]:
expected_result = "42"

def grade(workspace, artifact_path):
    workspace_files = []
    for path in workspace.iterdir():
        workspace_files.append(path.name)

    result_is_correct = artifact_path.read_text(encoding="utf-8") == expected_result
    workspace_is_clean = sorted(workspace_files) == ["numbers.txt"]
    return result_is_correct and workspace_is_clean

print("成功标准：结果等于 42，并且原工作区只保留 numbers.txt")

成功标准：结果等于 42，并且原工作区只保留 numbers.txt


输出显示了唯一的成功标准，说明后面的基线版本和改进版本会用同一把尺子判断结果。至此，工具、任务、工作区和成功标准都已准备完成，下一章将调用真实大模型并查看它返回的代码。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
模型、工具和任务已经准备完成。本节把它们发送给真实大模型，要求大模型必须选择工具，并记录等待回复所用的时间。

In [8]:
from time import perf_counter

# 记录一次真实 API 请求的等待时间
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实大模型已经返回结果，完整响应保存在 `response` 中。下一步会从响应里取出模型选择的工具和提交的 Python 代码。

## 3.2 查看并保存模型操作
API 响应包含多层数据，后续执行只需要工具名称和参数。本节读取第一条工具调用，把其中的 JSON 参数还原成 Python 数据，并保存模型生成的代码。

In [9]:
import json

# 从响应中取出模型提交的第一条工具调用
choice = response.choices[0]
tool_call = choice.message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)
tool_name = tool_call.function.name
python_code = arguments["code"]

print(f"工具：{tool_name}")
print("模型生成的代码：")
print(python_code)

工具：run_python
模型生成的代码：
# 读取 numbers.txt 中的两个整数
with open('numbers.txt', 'r') as f:
    content = f.read().strip()
    numbers = content.split()
    num1 = int(numbers[0])
    num2 = int(numbers[1])

# 计算两数之和
result = num1 + num2

# 将计算过程写入 scratch.txt
with open('scratch.txt', 'w') as f:
    f.write(f"读取的数字: {num1}, {num2}\n")
    f.write(f"计算过程: {num1} + {num2} = {result}\n")

# 将结果写入 result.txt
with open('result.txt', 'w') as f:
    f.write(str(result))

print(f"计算完成: {num1} + {num2} = {result}")
print("结果已写入 result.txt")


输出显示大模型选择了 `run_python`，并给出了读取输入、写入临时文件和最终文件的代码。`python_code` 已保存这段真实模型操作，后面的基线版本和改进版本将共同使用它。

## 3.3 查看本次请求信息
模型操作已经保存，还需要看清这次请求来自哪里、为何停止以及消耗了多少 Token。本节集中显示 provider、model、停止原因、Token 用量和等待时间。

In [10]:
# 读取真实响应附带的运行信息
usage = response.usage
stop_reason = choice.finish_reason

print(f"Provider：{config['NANO_BACKEND']}")
print(f"Model：{model_name}")
print(f"停止原因：{stop_reason}")
print(f"输入 Token：{usage.prompt_tokens}")
print(f"输出 Token：{usage.completion_tokens}")
print(f"总 Token：{usage.total_tokens}")
print(f"等待时间：{api_latency_ms} ms")

Provider：openai
Model：LongCat-2.0
停止原因：tool_calls
输入 Token：178
输出 Token：268
总 Token：446
等待时间：5140 ms


输出记录了本次真实 API 调用的来源、停止状态、Token 和延迟。停止原因是等待工具执行，不代表文件任务已经完成；下一章将定义直接在原工作区运行这段代码的基线组件。

# 4. 定义基线组件
## 4.1 定义直接执行器
最直接的做法是把模型生成的代码交给普通子进程，并把原工作区设为运行目录。这个执行器能完成任务，但代码产生的所有文件也会直接留在原工作区；本节先把这种做法保留为基线。

In [11]:
import subprocess
import sys

def run_directly(code, workspace):
    # 子进程直接使用原工作区，不复制文件，也不清理运行产物
    result = subprocess.run(
        [sys.executable, "-c", code],
        cwd=workspace,
        capture_output=True,
        text=True,
    )
    return result

print("基线组件：直接在原工作区运行代码")

基线组件：直接在原工作区运行代码


输出说明直接执行器已经定义，但模型代码尚未运行。下一章会把第 3 章保存的 `python_code` 交给它，并查看任务产物和临时文件是否都留在基线工作区。

# 5. 展示基线故障
## 5.1 直接执行模型代码
第 3 章已经保存了真实大模型生成的代码。现在把同一段代码交给直接执行器，让它在基线工作区中运行，并查看子进程是否正常结束。

In [12]:
# 直接在基线工作区运行真实模型生成的代码
baseline_result = run_directly(python_code, baseline_workspace)

print(f"退出码：{baseline_result.returncode}")
print("标准输出：")
print(baseline_result.stdout.strip())

退出码：0
标准输出：
计算完成: 20 + 22 = 42
结果已写入 result.txt


输出中的退出码为 `0`，说明代码正常运行并算出了结果。但进程成功不等于整个任务成功；下一步会查看这次运行给原工作区留下了哪些文件。

## 5.2 查看工作区变化
运行前，基线工作区中只有 `numbers.txt`。本节重新列出当前文件，并与第 2 章保存的初始列表比较，直接找出本次运行新增的内容。

In [13]:
# 收集运行后的全部文件名
baseline_files_after = []
for path in baseline_workspace.iterdir():
    baseline_files_after.append(path.name)

# 找出初始状态中不存在的文件
added_files = []
for file_name in baseline_files_after:
    if file_name not in baseline_files:
        added_files.append(file_name)

print(f"运行后文件：{sorted(baseline_files_after)}")
print(f"新增文件：{sorted(added_files)}")

运行后文件：['numbers.txt', 'result.txt', 'scratch.txt']
新增文件：['result.txt', 'scratch.txt']


输出显示 `result.txt` 和 `scratch.txt` 都直接留在了基线工作区。前者是需要交付的结果，后者只是运行过程中的临时文件；下一步先确认最终结果本身是否正确。

## 5.3 查看最终结果
工作区已经出现 `result.txt`。本节只读取这个文件的内容，确认模型生成的代码是否完成了加法任务。

In [14]:
# 读取基线版本生成的最终结果
baseline_artifact = baseline_workspace / "result.txt"
baseline_answer = baseline_artifact.read_text(encoding="utf-8")
print(f"最终结果：{baseline_answer}")

最终结果：42


输出是 `42`，说明大模型生成的代码和最终答案都正确。然而原工作区已经多出临时文件；下一步会使用第 2 章定义的完整标准判断这条执行路径。

## 5.4 判断基线结果
答案正确只是成功标准的一部分。本节把基线工作区和最终结果交给同一个 `grade()`，同时判断结果是否正确、原工作区是否仍保持初始状态。

In [15]:
# 使用第 2 章的共同标准判断基线结果
baseline_success = grade(baseline_workspace, baseline_artifact)
print(f"基线任务成功：{baseline_success}")

基线任务成功：False


输出为 `False`：模型代码正常结束，答案也是正确的，但直接执行污染了原工作区，因此完整任务仍然失败。下一章将定义 Sandbox，把同一段代码放进一次性工作区运行。

# 6. 定义改进组件
## 6.1 创建一次性工作区
现代代码 Agent 通常在一次性容器或 microVM 中执行任务。本 Notebook 为了用最少代码讲清核心机制，先把原工作区复制到独立的临时目录；后续所有写操作都会发生在这份副本中。

In [16]:
import shutil

def create_sandbox(source_workspace):
    # 创建一次性目录，再把原工作区复制进去
    sandbox_directory = TemporaryDirectory(prefix="nano_sandbox_run_")
    sandbox_workspace = Path(sandbox_directory.name) / "workspace"
    shutil.copytree(source_workspace, sandbox_workspace)
    return sandbox_directory, sandbox_workspace

print("改进组件 1：一次性工作区")

改进组件 1：一次性工作区


输出说明一次性工作区构件已经定义，但尚未复制文件。它会同时返回目录对象和工作区路径，目录对象将在任务结束时负责删除整份副本；下一步定义代码如何在副本中运行。

## 6.2 限制执行时间
一次性工作区限制了文件落在哪里，但进程仍可能一直运行。本节定义一个带时间上限的执行器：代码在指定工作区启动，超过给定秒数就停止等待。

In [17]:
def run_with_timeout(code, workspace, timeout_seconds):
    # timeout_seconds 给每次执行设置明确的最长时间
    result = subprocess.run(
        [sys.executable, "-c", code],
        cwd=workspace,
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )
    return result

print("改进组件 2：带时间上限的执行器")

改进组件 2：带时间上限的执行器


输出说明带时间上限的执行器已经定义，但尚未运行代码。下一步会定义产物出口，避免把一次性工作区中的临时文件全部带回。

## 6.3 只导出指定产物
Sandbox 内部可以产生许多文件，外部真正需要的通常只有少数交付物。本节定义一个选择性导出函数，只把明确指定的文件复制到产物目录。

In [18]:
def export_artifact(sandbox_workspace, artifact_directory, file_name):
    # 只复制调用者明确指定的一个产物
    source_path = sandbox_workspace / file_name
    artifact_path = artifact_directory / file_name
    shutil.copy2(source_path, artifact_path)
    return artifact_path

print("改进组件 3：选择性产物导出")

改进组件 3：选择性产物导出


输出说明三个改进构件都已准备好，但尚未执行任务。下一章会依次创建一次性工作区、运行同一段真实模型代码、只导出 `result.txt`，最后删除整个一次性工作区。

# 7. 展示修复结果
## 7.1 创建一次性工作区
改进版本从第 2 章准备的干净工作区开始。本节调用 `create_sandbox()`，把其中的 `numbers.txt` 复制到一份独立的临时副本。

In [19]:
# 从干净的改进工作区创建一次性副本
sandbox_directory, sandbox_workspace = create_sandbox(fixed_workspace)
print("一次性工作区已创建")

一次性工作区已创建


输出说明一次性工作区已经建立，原工作区和副本现在彼此独立。下一步会在副本中运行第 3 章保存的同一段真实模型代码。

## 7.2 在副本中限时执行
本节把 `python_code` 交给带时间上限的执行器，并把一次性工作区设为运行目录。执行时间上限设为 5 秒。

In [20]:
# 同一段真实模型代码只在一次性工作区中运行
fixed_result = run_with_timeout(
    python_code,
    sandbox_workspace,
    timeout_seconds=5,
)

print(f"退出码：{fixed_result.returncode}")
print("标准输出：")
print(fixed_result.stdout.strip())

退出码：0
标准输出：
计算完成: 20 + 22 = 42
结果已写入 result.txt


输出中的退出码为 `0`，说明同一段模型代码在一次性工作区中正常完成。下一步会查看它在副本内部产生了哪些文件。

## 7.3 查看副本内部状态
代码运行产生的文件此时都在一次性工作区中。本节列出副本内部的文件，直接观察最终结果和临时文件落在了哪里。

In [21]:
# 收集一次性工作区中的全部文件名
sandbox_files = []
for path in sandbox_workspace.iterdir():
    sandbox_files.append(path.name)

print(f"副本内部文件：{sorted(sandbox_files)}")

副本内部文件：['numbers.txt', 'result.txt', 'scratch.txt']


输出显示 `result.txt` 和 `scratch.txt` 都留在一次性副本中，没有直接写入原工作区。下一步只导出真正需要交付的 `result.txt`。

## 7.4 导出最终产物
副本中有最终结果和临时文件，但任务只要求交付 `result.txt`。本节明确指定这个文件，并把它复制到外部产物目录。

In [22]:
# 只把 result.txt 导出到外部产物目录
fixed_artifact = export_artifact(
    sandbox_workspace,
    artifact_directory,
    "result.txt",
)
print(f"已导出：{fixed_artifact.name}")

已导出：result.txt


输出说明 `result.txt` 已经离开一次性工作区，成为可保留的交付物。`scratch.txt` 没有被导出；下一步会删除包含它的整个副本。

## 7.5 删除一次性工作区
产物导出后，副本已经完成使命。本节调用临时目录自带的 `cleanup()`，一次删除副本中的输入、结果和临时文件。

In [23]:
# 删除整个一次性目录及其中的所有文件
sandbox_directory.cleanup()
print(f"一次性工作区仍存在：{sandbox_workspace.exists()}")

一次性工作区仍存在：False


输出为 `False`，说明一次性工作区已经被整体删除，内部的 `scratch.txt` 也随之消失。下一步会回到原工作区，查看它是否仍保持初始状态。

## 7.6 查看原工作区
Sandbox 的关键目标是让执行副作用停留在副本中。本节重新列出改进版本的原工作区文件，查看任务运行后它是否发生变化。

In [24]:
# 收集改进版本原工作区中的文件名
fixed_files_after = []
for path in fixed_workspace.iterdir():
    fixed_files_after.append(path.name)

print(f"原工作区文件：{sorted(fixed_files_after)}")

原工作区文件：['numbers.txt']


输出中仍然只有 `numbers.txt`，说明原工作区没有收到 `result.txt` 或 `scratch.txt`。下一步会查看外部产物目录实际保留了哪些文件。

## 7.7 查看导出目录
原工作区保持不变后，还要确认任务产物确实被保留下来。本节列出外部产物目录中的文件，观察选择性导出的结果。

In [25]:
# 收集外部产物目录中的文件名
exported_files = []
for path in artifact_directory.iterdir():
    exported_files.append(path.name)

print(f"导出目录文件：{sorted(exported_files)}")

导出目录文件：['result.txt']


输出中只有 `result.txt`，说明临时文件没有越过产物出口。下一步会读取这个文件，确认选择性导出没有改变最终答案。

## 7.8 查看最终结果
选择性导出控制的是文件范围，不应该改变文件内容。本节读取外部 `result.txt`，查看最终交付答案。

In [26]:
# 读取 Sandbox 导出的最终结果
fixed_answer = fixed_artifact.read_text(encoding="utf-8")
print(f"最终结果：{fixed_answer}")

最终结果：42


输出仍然是 `42`，说明改进版本既保留了正确结果，也隔离了运行副作用。下一步使用与基线版本相同的标准给出最终判断。

## 7.9 判断改进结果
本节把原工作区和导出的最终产物交给第 2 章定义的同一个 `grade()`。只有答案正确并且原工作区不变，改进路径才算成功。

In [27]:
# 使用与基线版本完全相同的成功标准
fixed_success = grade(fixed_workspace, fixed_artifact)
print(f"改进任务成功：{fixed_success}")

改进任务成功：True


输出为 `True`：同一段真实模型代码仍然得到正确答案，但临时文件只存在于已删除的一次性工作区，原工作区保持不变。下一章将汇总基线版本与改进版本的消融对照。

# 8. 汇总消融对照
## 8.1 汇总共同 API 信息
基线版本和改进版本复用了同一份真实模型代码，因此模型不是实验变量。本节先汇总这次共同 API 请求的 provider、model、停止原因、Token 和延迟。

In [28]:
# 保存两条执行路径共同使用的真实 API 信息
shared_api_info = {
    "Provider": config["NANO_BACKEND"],
    "Model": model_name,
    "真实 API 调用次数": 1,
    "停止原因": stop_reason,
    "总 Token": usage.total_tokens,
    "等待时间（毫秒）": api_latency_ms,
}
print(json.dumps(shared_api_info, ensure_ascii=False, indent=2))

{
  "Provider": "openai",
  "Model": "LongCat-2.0",
  "真实 API 调用次数": 1,
  "停止原因": "tool_calls",
  "总 Token": 446,
  "等待时间（毫秒）": 5140
}


输出说明两条路径共享同一次真实 API 调用和同一段模型代码。接下来只比较外层执行方式，避免把模型差异误认为 Sandbox 的效果。

## 8.2 对比两种执行方式
### 8.2.1 汇总对照数据
本节把两条路径的关键结果放进相同结构：原工作区文件、临时文件是否残留、交付位置、最终答案和任务是否成功。这里只整理已有结果，不重新执行任务。

In [29]:
# 使用相同字段汇总基线版本和改进版本
comparison = [
    {
        "执行方式": "直接执行",
        "原工作区文件": sorted(baseline_files_after),
        "临时文件残留": "scratch.txt" in baseline_files_after,
        "交付位置": "原工作区",
        "最终答案": baseline_answer,
        "任务成功": baseline_success,
    },
    {
        "执行方式": "Sandbox",
        "原工作区文件": sorted(fixed_files_after),
        "临时文件残留": "scratch.txt" in fixed_files_after,
        "交付位置": "产物目录",
        "最终答案": fixed_answer,
        "任务成功": fixed_success,
    },
]
print(f"已汇总 {len(comparison)} 条执行结果")

已汇总 2 条执行结果


输出说明两条执行结果已经使用相同字段整理完成。下一格只负责把这些数据完整展示出来。

### 8.2.2 展示对照数据
为了让文件状态和布尔结果一眼可见，本节把刚才整理的两条记录打印为缩进 JSON。

In [30]:
# 完整展示两种执行方式的结果
print(json.dumps(comparison, ensure_ascii=False, indent=2))

[
  {
    "执行方式": "直接执行",
    "原工作区文件": [
      "numbers.txt",
      "result.txt",
      "scratch.txt"
    ],
    "临时文件残留": true,
    "交付位置": "原工作区",
    "最终答案": "42",
    "任务成功": false
  },
  {
    "执行方式": "Sandbox",
    "原工作区文件": [
      "numbers.txt"
    ],
    "临时文件残留": false,
    "交付位置": "产物目录",
    "最终答案": "42",
    "任务成功": true
  }
]


输出显示两种方式都得到答案 `42`。直接执行把临时文件留在原工作区，任务失败；Sandbox 只保留外部产物，原工作区不变，任务成功。下一步用一个状态变化总结消融结果。

## 8.3 总结机制效果
模型、代码、输入和成功标准都没有改变，唯一变化是外层执行方式。本节直接打印基线与改进版本的成功状态变化。

In [31]:
# 只比较加入 Sandbox 前后的任务成功状态
success_change = f"{baseline_success} -> {fixed_success}"
print(f"任务成功变化：{success_change}")
print("唯一改变：直接执行 -> Sandbox")

任务成功变化：False -> True
唯一改变：直接执行 -> Sandbox


输出中的 `False -> True` 说明：模型本来就能生成正确代码，真正决定完整任务是否成功的是外层执行边界。一次性工作区、时间上限和选择性产物导出共同把运行副作用留在可删除的环境中，至此本 Notebook 的消融对照结束。

## 8.4 拓展

### nano 版省略了什么

nano 版使用本机临时目录和进程超时，没有真正的用户、内核、网络、系统调用、密钥与资源隔离，也未覆盖镜像供应链、逃逸检测、WASM 或微虚拟机。生产 Sandbox 还需默认拒绝网络、限制 CPU/内存/进程数、净化环境变量并验证导出产物；本例只证明副作用必须被关在可重置边界内。

### 延伸阅读

1. 2025, [OpenAI, Introducing Codex](https://openai.com/index/introducing-codex/)：云端编码 Agent 的隔离环境、任务执行与结果审阅。
2. 2025, [Anthropic, Beyond permission prompts](https://www.anthropic.com/engineering/claude-code-sandboxing)：面向 Agent 的文件系统与网络沙箱实践。
3. 2024, [NVIDIA, Sandboxing Agentic AI Workflows with WebAssembly](https://developer.nvidia.com/blog/sandboxing-agentic-ai-workflows-with-webassembly/)：WASM 能力边界与可移植隔离方案。